[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/data-engineering-from-scratch-to-advanced/blob/main/notebooks/topics/13_airflow.ipynb)

*Data Engineering from Scratch to Advanced · Notebook 13 of 21*

# Orchestration With Airflow

Running pipelines reliably: DAGs and topological order, schedules and data intervals, catchup and backfills, retries, sensors, XComs, pools and common mistakes. You'll build a tiny scheduler to see how it works.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** Airflow is a conductor, not a musician. It tells others when to play, and doesn't do the heavy work itself.

**What interviewers really test here:** DAGs, the logical date, catchup, idempotent tasks and common operational mistakes.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Why you need an orchestrator *(Basics)*
- [ ] 2. Dates: the idea that confuses everyone *(Core)*
- [ ] 3. Failures, retries and dependencies *(Core)*
- [ ] 4. Airflow mistakes and alternatives *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import numpy as np
import matplotlib.pyplot as plt
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Core concepts

| Term | Meaning |
| --- | --- |
| DAG | tasks and their dependencies |
| Task / operator | one unit of work / a template for one |
| DAG run | one execution, for one logical date |
| Sensor | a task that waits for a condition |
| XCom | small values passed between tasks |
| Pool | a concurrency limit for a shared resource |
| Connection | stored credentials for an external system |

---
# Part 2: Lessons, step by step

## Lesson 1: Why you need an orchestrator

*Level: Basics*

Your pipeline has steps: extract, then transform, then load, then a report. Cron can run a script at 2 am, but who handles the rest?



| Question | Cron | Airflow |
| --- | --- | --- |
| Run in dependency order? | you guess the timing | yes |
| Retry a failure? | no | yes |
| Rerun last Tuesday? | by hand | one click |
| See what ran, and alert when it fails? | no | yes |



**A DAG** (directed acyclic graph) describes the tasks and their order. Directed means arrows have a direction. Acyclic means there are no loops, as A waiting for B while B waits for A could never finish.



```text
extract ─┬→ transform ─→ load
         └→ audit
```



**Topological sort** finds a valid order: extract, audit, transform, load. Tasks in the same **level** can run at once.



**Critical path.** With tasks a (1 s), then b (5 s) or c (2 s), then d (1 s): even with unlimited workers it takes 1 + 5 + 1 = **7 s**. Speeding up c gains nothing. Only the longest path matters.



```python
from airflow.decorators import dag, task
from datetime import datetime

@dag(schedule="@daily", start_date=datetime(2026, 1, 1), catchup=False)
def orders_pipeline():

    @task(retries=3)
    def extract(ds=None):                 # ds: the date this run is for
        path = f"s3://lake/raw/orders/{ds}.json"
        ...
        return path                       # pass the path, not the data

    @task
    def load(path):
        ...

    load(extract())

orders_pipeline()
```



Airflow doesn't run in this site's browser, so questions here build a tiny scheduler in Python, which shows you exactly what Airflow does.

**✏️ Your turn:** you now have what you need for [Q9](#q9), [Q10](#q10), [Q11](#q11), [Q18](#q18), [Q8](#q8). Try one now, or carry on and do them in Part 3.

## Lesson 2: Dates: the idea that confuses everyone

*Level: Core*

**A daily DAG's run for 5 January starts on 6 January.** That's not a bug. It processes the data *for* 5 January, which isn't complete until the day ends.



| Term | Value |
| --- | --- |
| Logical date (`ds`) | 2026-01-05 |
| Data interval | 05 Jan 00:00 to 06 Jan 00:00 |
| Actually runs at | 06 Jan 00:00 |



**Always use the logical date in your code, never "now".** Then rerunning 5 January a month later reads the right data. This is what makes runs idempotent.



```python
s3://lake/dt={{ ds }}/part-{{ ds_nodash }}.parquet
→ s3://lake/dt=2026-01-05/part-20260105.parquet
```



**Catchup.** Deploy a daily DAG with a start date 10 days ago, with catchup on, and Airflow immediately creates **10 runs**. It has surprised many people, and has hammered many databases. Set `catchup=False` unless you want that.



**Backfills** rerun history deliberately. 30 days at 10 minutes each, with at most 5 at once, takes 60 minutes. Limit the concurrency, so you don't overload what's downstream.

**✏️ Your turn:** you now have what you need for [Q12](#q12), [Q1](#q1), [Q7](#q7), [Q16](#q16). Try one now, or carry on and do them in Part 3.

## Lesson 3: Failures, retries and dependencies

*Level: Core*

**Retries:** a task's states go queued, running, then success or failed. With 3 retries, it gets 4 tries. Use growing delays, such as 60, 120, 240 seconds, as a struggling service needs time.



**Trigger rules** decide if a task runs, given what happened upstream:



| Rule | Runs when | Use for |
| --- | --- | --- |
| `all_success` (default) | all upstream succeeded | normal steps |
| `all_done` | all finished, however they ended | clean-up |
| `one_failed` | any failed | send an alert |



If b fails in a → b → c, then c is marked **upstream\_failed**, not run.



**Clearing a task** reruns it and everything downstream. Clear b in a→b→c, a→d, and b and c rerun, while d stays as it is.



**Sensors** wait for something, like a file. Checking every 30 s for a file that shows up at 90 s means 4 pokes. Waiting sensors can hold a worker slot, so use `mode="reschedule"` or deferrable versions for long waits.



**Pools** limit concurrency for a shared resource. Tasks 4, 2, 8, 3 with 2 slots finish at 10.



**SLAs and anomaly detection.** Alert on lateness: a task finishing at 8:30 for an 8:00 promise is 30 minutes late. Past runs give another signal: durations 100, 110, 90, 100 set a limit around 121, so 200 is worth a look even if nothing failed.

**✏️ Your turn:** you now have what you need for [Q2](#q2), [Q3](#q3), [Q13](#q13), [Q14](#q14), [Q4](#q4), [Q19](#q19), [Q5](#q5), [Q17](#q17), [Q20](#q20). Try one now, or carry on and do them in Part 3.

## Lesson 4: Airflow mistakes and alternatives

*Level: Advanced*

**Airflow should orchestrate, not process.** Have it tell Spark or a warehouse what to do. Don't load 10 GB into a Python task.



**Common mistakes:**



| Mistake | Why it hurts | Do this instead |
| --- | --- | --- |
| Heavy code at the top level of a DAG file | the scheduler parses files constantly, so it runs every time | put it inside tasks |
| Big data in XCom | it's stored in the metadata database | write to storage, pass the path |
| Using "now" in tasks | reruns get the wrong data | use the logical date |
| Non-idempotent tasks | retries duplicate data | overwrite or upsert |
| Dynamic start\_date | confuses scheduling | use a fixed date |



**Parts of Airflow:** the **scheduler** decides what's ready; **workers** run tasks; the **metadata database** stores state; and a **web server** shows it. Executors include Local, Celery and Kubernetes.



| Tool | Focus |
| --- | --- |
| Airflow | task-centred, the industry default |
| Dagster | asset-centred, focused on the data produced |
| Prefect | Python-native flows |
| Cloud (Step Functions, ADF) | tightly integrated, less portable |

**✏️ Your turn:** you now have what you need for [Q6](#q6), [Q15](#q15). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [2]:
from datetime import date, datetime, timedelta

from collections import defaultdict, deque

import heapq


<a id="q1"></a>
### Q1. How Many Runs Will Catchup Create?

🟢 Warm-up · *A classic surprise*

With a daily DAG and catchup on, return the runs created from start\_date up to today (the interval must have ended). With catchup off, it's at most 1.

**Examples**

- `start = '2026-01-01', today = '2026-01-11', catchup = True` → `10`
- `start = '2026-01-01', today = '2026-01-11', catchup = False` → `1`
- `start = '2026-01-11', today = '2026-01-11', catchup = True` → `0`

In [ ]:
def catchup_runs(start, today, catchup):
    """
    With a daily DAG and catchup on, return the runs created from start_date up
    to today (the interval must have ended). With catchup off, it's at most 1.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(catchup_runs, [
    (('2026-01-01', '2026-01-11', True), 10),
    (('2026-01-01', '2026-01-11', False), 1),
    (('2026-01-11', '2026-01-11', True), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop through dates.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Date subtraction.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Loop through dates.

```python
def catchup_runs(start, today, catchup):
    d = date.fromisoformat(start)
    end = date.fromisoformat(today)
    count = 0
    while d + timedelta(days=1) <= end:
        count += 1
        d += timedelta(days=1)
    return count if catchup else min(count, 1)
```

**Cost:** O(d) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Date subtraction.

```python
def catchup_runs(start, today, catchup):
    n = max(0, (date.fromisoformat(today) - date.fromisoformat(start)).days)
    return n if catchup else min(n, 1)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop through dates. That costs O(d) time and O(1) space."

"I can do better: Date subtraction. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/airflow/how-many-runs-will-catchup-create)

<a id="q2"></a>
### Q2. Retries With Exponential Delay

🟢 Warm-up · *When does each retry happen?*

Return the retry times in seconds after failing at t0: delay, 2×delay, 4×delay… added cumulatively, capped at max per wait.

**Examples**

- `t0 = 0, delay = 60, retries = 4, cap = 300` → `[60, 180, 420, 720]`

In [ ]:
def retry_times(t0, delay, retries, cap):
    """
    Return the retry times in seconds after failing at t0: delay, 2×delay,
    4×delay… added cumulatively, capped at max per wait.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(retry_times, [
    ((0, 60, 4, 300), [60, 180, 420, 720]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`itertools.accumulate`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop.

```python
def retry_times(t0, delay, retries, cap):
    out, t = [], t0
    for i in range(retries):
        t += min(delay * 2 ** i, cap)
        out.append(t)
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `itertools.accumulate`.

```python
def retry_times(t0, delay, retries, cap):
    import itertools
    return list(itertools.accumulate((min(delay * 2 ** i, cap) for i in range(retries)), initial=t0))[1:]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop. That costs O(n) time and O(n) space."

"I can do better: `itertools.accumulate`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/airflow/retries-with-exponential-delay)

<a id="q3"></a>
### Q3. Task State After Attempts

🟢 Warm-up · *Success, retry or failed*

attempts are True or False. With retries r, return the final state: 'success', or 'failed' if all r + 1 attempts fail, and attempts used.

**Examples**

- `attempts = [False, True], r = 3` → `('success', 2)`
- `attempts = [False, False, False], r = 2` → `('failed', 3)`

In [ ]:
def task_state(attempts, r):
    """
    attempts are True or False. With retries r, return the final state:
    'success', or 'failed' if all r + 1 attempts fail, and attempts used.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(task_state, [
    (([False, True], 3), ('success', 2)),
    (([False, False, False], 2), ('failed', 3)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Slice, then `index`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop.

```python
def task_state(attempts, r):
    used = 0
    for ok in attempts[:r + 1]:
        used += 1
        if ok:
            return ("success", used)
    return ("failed", used)
```

**Cost:** O(r) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Slice, then `index`.

```python
def task_state(attempts, r):
    tries = attempts[:r + 1]
    return ("success", tries.index(True) + 1) if True in tries else ("failed", len(tries))
```

**Cost:** O(r) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop. That costs O(r) time and O(1) space."

"I can do better: Slice, then `index`. That's O(r) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/airflow/task-state-after-attempts)

<a id="q4"></a>
### Q4. Sensor: Wait for a File

🟢 Warm-up · *Poke until ready*

A sensor checks every interval until the file exists at ready\_at, or fails after timeout. Return ('success' or 'timeout', number of pokes). Checks start at 0.

**Examples**

- `ready_at = 90, interval = 30, timeout = 300` → `('success', 4)`
- `ready_at = 1000, interval = 30, timeout = 90` → `('timeout', 4)`

In [ ]:
def sensor(ready_at, interval, timeout):
    """
    A sensor checks every interval until the file exists at ready_at, or fails
    after timeout. Return ('success' or 'timeout', number of pokes). Checks
    start at 0.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sensor, [
    ((90, 30, 300), ('success', 4)),
    ((1000, 30, 90), ('timeout', 4)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Simulate the checks.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Integer arithmetic.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Simulate the checks.

```python
def sensor(ready_at, interval, timeout):
    t, pokes = 0, 0
    while t <= timeout:
        pokes += 1
        if t >= ready_at:
            return ("success", pokes)
        t += interval
    return ("timeout", pokes)
```

**Cost:** O(t/i) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Integer arithmetic.

```python
def sensor(ready_at, interval, timeout):
    needed = -(-ready_at // interval)
    if needed * interval <= timeout:
        return ("success", needed + 1)
    return ("timeout", timeout // interval + 1)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Simulate the checks. That costs O(t/i) time and O(1) space."

"I can do better: Integer arithmetic. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/airflow/sensor-wait-for-a-file)

<a id="q5"></a>
### Q5. Is the SLA Met?

🟢 Warm-up · *Alert before anyone complains*

Given a task's start and duration and an SLA time, return (finish, minutes late, or 0).

**Examples**

- `start = 360, duration = 90, sla = 480` → `(450, 0)`
- `start = 360, duration = 150, sla = 480` → `(510, 30)`

In [ ]:
def sla_check(start, duration, sla):
    """
    Given a task's start and duration and an SLA time, return (finish, minutes
    late, or 0).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sla_check, [
    ((360, 90, 480), (450, 0)),
    ((360, 150, 480), (510, 30)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute and compare.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`max`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Compute and compare.

```python
def sla_check(start, duration, sla):
    finish = start + duration
    late = 0
    if finish > sla:
        late = finish - sla
    return (finish, late)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `max`.

```python
def sla_check(start, duration, sla):
    return (start + duration, max(0, start + duration - sla))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute and compare. That costs O(1) time and O(1) space."

"I can do better: `max`. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/airflow/is-the-sla-met)

<a id="q6"></a>
### Q6. Don't Pass Data Through XCom

🟢 Warm-up · *Pass a reference*

XCom stores small values in the metadata database. Return whether a payload in KB is acceptable under a limit, and otherwise suggest 'write to storage and pass the path'.

**Examples**

- `size_kb = 2, limit_kb = 48` → `'ok'`
- `size_kb = 5000, limit_kb = 48` → `'write to storage and pass the path'`

In [ ]:
def xcom_ok(size_kb, limit_kb):
    """
    XCom stores small values in the metadata database. Return whether a payload
    in KB is acceptable under a limit, and otherwise suggest 'write to storage
    and pass the path'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(xcom_ok, [
    ((2, 48), 'ok'),
    ((5000, 48), 'write to storage and pass the path'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

An if.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A conditional expression.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** An if.

```python
def xcom_ok(size_kb, limit_kb):
    if size_kb <= limit_kb:
        return "ok"
    return "write to storage and pass the path"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A conditional expression.

```python
def xcom_ok(size_kb, limit_kb):
    return "ok" if size_kb <= limit_kb else "write to storage and pass the path"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: An if. That costs O(1) time and O(1) space."

"I can do better: A conditional expression. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/airflow/dont-pass-data-through-xcom)

<a id="q7"></a>
### Q7. Render a Date Template

🟢 Warm-up · *Jinja in Airflow*

Replace {{ ds }} with the logical date, and {{ ds\_nodash }} with it without dashes.

**Examples**

- `template = 's3://lake/dt={{ ds }}/part-{{ ds_nodash }}.parquet', ds = '2026-01-05'` → `'s3://lake/dt=2026-01-05/part-20260105.parquet'`

In [ ]:
def render(template, ds):
    """
    Replace {{ ds }} with the logical date, and {{ ds_nodash }} with it without
    dashes.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(render, [
    (('s3://lake/dt={{ ds }}/part-{{ ds_nodash }}.parquet', '2026-01-05'), 's3://lake/dt=2026-01-05/part-20260105.parquet'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Two replaces.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A regular expression with a dictionary.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Two replaces.

```python
def render(template, ds):
    return template.replace("{{ ds }}", ds).replace("{{ ds_nodash }}", ds.replace("-", ""))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A regular expression with a dictionary.

```python
def render(template, ds):
    import re
    values = {"ds": ds, "ds_nodash": ds.replace("-", "")}
    return re.sub(r"\{\{\s*(\w+)\s*\}\}", lambda m: values[m.group(1)], template)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Two replaces. That costs O(n) time and O(n) space."

"I can do better: A regular expression with a dictionary. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/airflow/render-a-date-template)

<a id="q8"></a>
### Q8. Which Tasks Are Ready?

🟢 Warm-up · *The scheduler's core loop*

Given dependencies and finished tasks, return tasks not done whose upstream are all done, sorted.

**Examples**

- `edges = [('a', 'b'), ('a', 'c'), ('b', 'd')], tasks = ['a', 'b', 'c', 'd'], done = ['a']` → `['b', 'c']`

In [ ]:
def ready_tasks(edges, tasks, done):
    """
    Given dependencies and finished tasks, return tasks not done whose upstream
    are all done, sorted.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(ready_tasks, [
    (([('a', 'b'), ('a', 'c'), ('b', 'd')], ['a', 'b', 'c', 'd'], ['a']), ['b', 'c']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop over tasks and edges.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A set of finished tasks.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop over tasks and edges.

```python
def ready_tasks(edges, tasks, done):
    out = []
    for t in tasks:
        if t not in done and all(a in done for a, b in edges if b == t):
            out.append(t)
    return sorted(out)
```

**Cost:** O(V·E) time, O(V) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A set of finished tasks.

```python
def ready_tasks(edges, tasks, done):
    d = set(done)
    blocked = {b for a, b in edges if a not in d}
    return sorted(set(tasks) - d - blocked)
```

**Cost:** O(V + E) time, O(V) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop over tasks and edges. That costs O(V·E) time and O(V) space."

"I can do better: A set of finished tasks. That's O(V + E) time and O(V) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/airflow/which-tasks-are-ready)

<a id="q9"></a>
### Q9. Is This a Valid DAG?

🟡 Interview standard · *No cycles allowed*

edges are (before, after). Return whether the graph has no cycle.

**Examples**

- `edges = [('a', 'b'), ('b', 'c')]` → `True`
- `edges = [('a', 'b'), ('b', 'a')]` → `False`

In [ ]:
def is_dag(edges):
    """
    edges are (before, after). Return whether the graph has no cycle.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(is_dag, [
    (([('a', 'b'), ('b', 'c')],), True),
    (([('a', 'b'), ('b', 'a')],), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Repeatedly remove nodes with no incoming edges.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Kahn's algorithm with indegree counts.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Repeatedly remove nodes with no incoming edges.

```python
def is_dag(edges):
    remaining = list(edges)
    nodes = {n for e in edges for n in e}
    while nodes:
        free = {n for n in nodes if not any(b == n for _, b in remaining)}
        if not free:
            return False
        nodes -= free
        remaining = [e for e in remaining if e[0] not in free]
    return True
```

**Cost:** O(V·E) time, O(V) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Kahn's algorithm with indegree counts.

```python
def is_dag(edges):
    after, indeg = defaultdict(list), defaultdict(int)
    nodes = set()
    for a, b in edges:
        after[a].append(b)
        indeg[b] += 1
        nodes |= {a, b}
    ready = deque(n for n in nodes if indeg[n] == 0)
    seen = 0
    while ready:
        n = ready.popleft()
        seen += 1
        for m in after[n]:
            indeg[m] -= 1
            if indeg[m] == 0:
                ready.append(m)
    return seen == len(nodes)
```

**Cost:** O(V + E) time, O(V + E) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Repeatedly remove nodes with no incoming edges. That costs O(V·E) time and O(V) space."

"I can do better: Kahn's algorithm with indegree counts. That's O(V + E) time and O(V + E) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/airflow/is-this-a-valid-dag)

<a id="q10"></a>
### Q10. Run Order

🟡 Interview standard · *Topological sort*

Return tasks in an order that respects dependencies; when several are ready, choose alphabetically.

**Examples**

- `edges = [('extract', 'transform'), ('transform', 'load'), ('extract', 'audit')]` → `['extract', 'audit', 'transform', 'load']`

In [ ]:
def run_order(edges):
    """
    Return tasks in an order that respects dependencies; when several are ready,
    choose alphabetically.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(run_order, [
    (([('extract', 'transform'), ('transform', 'load'), ('extract', 'audit')],), ['extract', 'audit', 'transform', 'load']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Repeatedly scan for ready tasks.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Kahn's algorithm with a heap.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Repeatedly scan for ready tasks.

```python
def run_order(edges):
    nodes = {n for e in edges for n in e}
    order = []
    while nodes:
        ready = sorted(n for n in nodes if not any(b == n and a in nodes for a, b in edges))
        order.append(ready[0])
        nodes.remove(ready[0])
    return order
```

**Cost:** O(V²) time, O(V) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Kahn's algorithm with a heap.

```python
def run_order(edges):
    after, indeg = defaultdict(list), defaultdict(int)
    nodes = set()
    for a, b in edges:
        after[a].append(b)
        indeg[b] += 1
        nodes |= {a, b}
    heap = sorted(n for n in nodes if indeg[n] == 0)
    order = []
    while heap:
        n = heapq.heappop(heap)
        order.append(n)
        for m in after[n]:
            indeg[m] -= 1
            if not indeg[m]:
                heapq.heappush(heap, m)
    return order
```

**Cost:** O((V + E) log V) time, O(V) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Repeatedly scan for ready tasks. That costs O(V²) time and O(V) space."

"I can do better: Kahn's algorithm with a heap. That's O((V + E) log V) time and O(V) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/airflow/run-order)

<a id="q11"></a>
### Q11. Parallel Levels

🟡 Interview standard · *What can run at once?*

Group tasks into levels: each runs after all its dependencies. Return levels, each sorted.

**Examples**

- `edges = [('a', 'c'), ('b', 'c'), ('c', 'd'), ('c', 'e')]` → `[['a', 'b'], ['c'], ['d', 'e']]`

In [ ]:
def levels(edges):
    """
    Group tasks into levels: each runs after all its dependencies. Return
    levels, each sorted.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(levels, [
    (([('a', 'c'), ('b', 'c'), ('c', 'd'), ('c', 'e')],), [['a', 'b'], ['c'], ['d', 'e']]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Repeatedly take all ready tasks.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Kahn's algorithm, one level at a time.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Repeatedly take all ready tasks.

```python
def levels(edges):
    nodes = {n for e in edges for n in e}
    done, out = set(), []
    while nodes - done:
        level = sorted(n for n in nodes - done if all(a in done for a, b in edges if b == n))
        out.append(level)
        done |= set(level)
    return out
```

**Cost:** O(V²) time, O(V) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Kahn's algorithm, one level at a time.

```python
def levels(edges):
    after, indeg = defaultdict(list), defaultdict(int)
    nodes = set()
    for a, b in edges:
        after[a].append(b)
        indeg[b] += 1
        nodes |= {a, b}
    level, out = sorted(n for n in nodes if not indeg[n]), []
    while level:
        out.append(level)
        nxt = []
        for n in level:
            for m in after[n]:
                indeg[m] -= 1
                if not indeg[m]:
                    nxt.append(m)
        level = sorted(nxt)
    return out
```

**Cost:** O(V + E) time, O(V) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Repeatedly take all ready tasks. That costs O(V²) time and O(V) space."

"I can do better: Kahn's algorithm, one level at a time. That's O(V + E) time and O(V) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/airflow/parallel-levels)

<a id="q12"></a>
### Q12. Daily Data Intervals

🟡 Interview standard · *Airflow's most confusing idea*

A daily DAG's run for an interval starts after the interval ends. Given the logical date, return (interval start, interval end, when it actually runs). Times are days.

**Examples**

- `logical = '2026-01-05'` → `('2026-01-05', '2026-01-06', '2026-01-06')`

In [ ]:
def data_interval(logical):
    """
    A daily DAG's run for an interval starts after the interval ends. Given the
    logical date, return (interval start, interval end, when it actually runs).
    Times are days.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(data_interval, [
    (('2026-01-05',), ('2026-01-05', '2026-01-06', '2026-01-06')),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Add one day by hand.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compact.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Add one day by hand.

```python
def data_interval(logical):
    d = date.fromisoformat(logical)
    end = d + timedelta(days=1)
    return (d.isoformat(), end.isoformat(), end.isoformat())
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Compact.

```python
def data_interval(logical):
    d = date.fromisoformat(logical)
    e = (d + timedelta(days=1)).isoformat()
    return (logical, e, e)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Add one day by hand. That costs O(1) time and O(1) space."

"I can do better: Compact. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/airflow/daily-data-intervals)

<a id="q13"></a>
### Q13. Upstream Failed

🟡 Interview standard · *Trigger rules*

Given each task's upstream states, return whether it runs. Rules: 'all\_success', 'all\_done', 'one\_failed'.

**Examples**

- `upstream = ['success', 'failed'], rule = 'all_success'` → `False`
- `upstream = ['success', 'failed'], rule = 'all_done'` → `True`
- `upstream = ['success', 'success'], rule = 'one_failed'` → `False`

In [ ]:
def should_run(upstream, rule):
    """
    Given each task's upstream states, return whether it runs. Rules:
    'all_success', 'all_done', 'one_failed'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(should_run, [
    ((['success', 'failed'], 'all_success'), False),
    ((['success', 'failed'], 'all_done'), True),
    ((['success', 'success'], 'one_failed'), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Ifs.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dictionary of rules.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Ifs.

```python
def should_run(upstream, rule):
    if rule == "all_success":
        return all(s == "success" for s in upstream)
    if rule == "one_failed":
        return "failed" in upstream
    return True
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A dictionary of rules.

```python
def should_run(upstream, rule):
    return {"all_success": set(upstream) <= {"success"}, "one_failed": "failed" in upstream, "all_done": True}[rule]
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Ifs. That costs O(n) time and O(1) space."

"I can do better: A dictionary of rules. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/airflow/upstream-failed)

<a id="q14"></a>
### Q14. What Needs Rerunning?

🟡 Interview standard · *Clear downstream*

If a task is cleared, it and all downstream tasks rerun. Return them, sorted.

**Examples**

- `edges = [('a', 'b'), ('b', 'c'), ('a', 'd')], task = 'b'` → `['b', 'c']`

In [ ]:
def to_rerun(edges, task):
    """
    If a task is cleared, it and all downstream tasks rerun. Return them,
    sorted.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(to_rerun, [
    (([('a', 'b'), ('b', 'c'), ('a', 'd')], 'b'), ['b', 'c']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Repeat until no change.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Breadth-first search.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Repeat until no change.

```python
def to_rerun(edges, task):
    found = {task}
    changed = True
    while changed:
        changed = False
        for a, b in edges:
            if a in found and b not in found:
                found.add(b)
                changed = True
    return sorted(found)
```

**Cost:** O(V·E) time, O(V) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Breadth-first search.

```python
def to_rerun(edges, task):
    after = defaultdict(list)
    for a, b in edges:
        after[a].append(b)
    seen, todo = {task}, deque([task])
    while todo:
        for m in after[todo.popleft()]:
            if m not in seen:
                seen.add(m)
                todo.append(m)
    return sorted(seen)
```

**Cost:** O(V + E) time, O(V) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Repeat until no change. That costs O(V·E) time and O(V) space."

"I can do better: Breadth-first search. That's O(V + E) time and O(V) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/airflow/what-needs-rerunning)

<a id="q15"></a>
### Q15. Find Top-Level Work in a DAG File

🟡 Interview standard · *The scheduler parses it constantly*

Top-level code runs on every parse. Given a DAG file's lines, return unindented lines that call a function other than dag(...) or a decorator, using a simple test: not import/from/def/@/#, and contains '('.

**Examples**

- `lines = ['import x', 'rows = fetch_all()', '@dag()', 'def d():', '    y = fetch()', 'd()']` → `['rows = fetch_all()', 'd()']`

In [ ]:
def top_level_calls(lines):
    """
    Top-level code runs on every parse. Given a DAG file's lines, return
    unindented lines that call a function other than dag(...) or a decorator,
    using a simple test: not import/from/def/@/#, and contains '('.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(top_level_calls, [
    ((['import x', 'rows = fetch_all()', '@dag()', 'def d():', '    y = fetch()', 'd()'],), ['rows = fetch_all()', 'd()']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Checks in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A regular expression.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Checks in a loop.

```python
def top_level_calls(lines):
    out = []
    for line in lines:
        if line[:1] in (" ", "\t", "") or "(" not in line:
            continue
        if line.startswith(("import", "from", "def", "@", "#")):
            continue
        out.append(line)
    return out
```

**Cost:** O(n) time, O(m) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A regular expression.

```python
def top_level_calls(lines):
    import re
    return [l for l in lines if re.match(r"(?!import|from|def|@|#)\S.*\(", l)]
```

**Cost:** O(n) time, O(m) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Checks in a loop. That costs O(n) time and O(m) space."

"I can do better: A regular expression. That's O(n) time and O(m) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/airflow/find-top-level-work-in-a-dag-file)

<a id="q16"></a>
### Q16. Backfill With a Concurrency Limit

🟡 Interview standard · *Don't overload*

Backfill d days, each taking m minutes, with at most c at once. Return total minutes.

**Examples**

- `d = 30, m = 10, c = 5` → `60`
- `d = 31, m = 10, c = 5` → `70`

In [ ]:
def backfill_minutes(d, m, c):
    """
    Backfill d days, each taking m minutes, with at most c at once. Return total
    minutes.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(backfill_minutes, [
    ((30, 10, 5), 60),
    ((31, 10, 5), 70),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Simulate rounds.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Round up.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Simulate rounds.

```python
def backfill_minutes(d, m, c):
    left, total = d, 0
    while left > 0:
        left -= c
        total += m
    return total
```

**Cost:** O(d/c) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Round up.

```python
def backfill_minutes(d, m, c):
    return -(-d // c) * m
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Simulate rounds. That costs O(d/c) time and O(1) space."

"I can do better: Round up. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/airflow/backfill-with-a-concurrency-limit)

<a id="q17"></a>
### Q17. Duration Anomaly

🟡 Interview standard · *Spot slowdowns early*

Given past durations and today's, return whether today exceeds mean + 3 standard deviations (population), and the threshold, rounded to 2 decimals.

**Examples**

- `history = [100, 110, 90, 100], today = 200` → `(True, 121.21)`
- `history = [100, 110, 90, 100], today = 105` → `(False, 121.21)`

In [ ]:
def duration_alert(history, today):
    """
    Given past durations and today's, return whether today exceeds mean + 3
    standard deviations (population), and the threshold, rounded to 2 decimals.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(duration_alert, [
    (([100, 110, 90, 100], 200), (True, 121.21)),
    (([100, 110, 90, 100], 105), (False, 121.21)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute the mean and deviation by hand.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`statistics.pstdev`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Compute the mean and deviation by hand.

```python
def duration_alert(history, today):
    mean = sum(history) / len(history)
    variance = sum((x - mean) ** 2 for x in history) / len(history)
    limit = mean + 3 * variance ** 0.5
    return (today > limit, round(limit, 2))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `statistics.pstdev`.

```python
def duration_alert(history, today):
    import statistics
    limit = statistics.mean(history) + 3 * statistics.pstdev(history)
    return (today > limit, round(limit, 2))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute the mean and deviation by hand. That costs O(n) time and O(1) space."

"I can do better: `statistics.pstdev`. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/airflow/duration-anomaly)

<a id="q18"></a>
### Q18. Critical Path

🔴 Stretch · *The minimum time for a DAG*

durations maps task to seconds. Return the total time with unlimited parallelism: the longest path through the graph.

**Examples**

- `edges = [('a', 'b'), ('a', 'c'), ('b', 'd'), ('c', 'd')], durations = {'a': 1, 'b': 5, 'c': 2, 'd': 1}` → `7`

In [ ]:
def critical_path(edges, durations):
    """
    durations maps task to seconds. Return the total time with unlimited
    parallelism: the longest path through the graph.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(critical_path, [
    (([('a', 'b'), ('a', 'c'), ('b', 'd'), ('c', 'd')], {'a': 1, 'b': 5, 'c': 2, 'd': 1}), 7),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every path recursively.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Dynamic programming in topological order.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Try every path recursively.

```python
def critical_path(edges, durations):
    def longest(n):
        before = [a for a, b in edges if b == n]
        return durations[n] + max((longest(p) for p in before), default=0)
    return max(longest(n) for n in durations)
```

**Cost:** exponential worst case time, O(V) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Dynamic programming in topological order.

```python
def critical_path(edges, durations):
    import functools
    before = defaultdict(list)
    for a, b in edges:
        before[b].append(a)

    @functools.lru_cache(maxsize=None)
    def finish(n):
        return durations[n] + max(map(finish, before[n]), default=0)
    return max(map(finish, durations))
```

**Cost:** O(V + E) time, O(V) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every path recursively. That costs exponential worst case time and O(V) space."

"I can do better: Dynamic programming in topological order. That's O(V + E) time and O(V) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `edges = [('a', 'b'), ('a', 'c'), ('b', 'd'), ('c', 'd')], durations = {'a': 1, 'b': 5, 'c': 2, 'd': 1}`

1. finish() returns 1
2. finish() returns 6
3. finish() returns 3
4. finish() returns 7
5. critical\_path() returns 7

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/airflow/critical-path)

<a id="q19"></a>
### Q19. Pool Scheduling

🔴 Stretch · *Limit concurrency*

Tasks have durations and run in order, using at most slots at once. Return the finish time.

**Examples**

- `tasks = [4, 2, 8, 3], slots = 2` → `10`
- `tasks = [1, 1, 1], slots = 5` → `1`

In [ ]:
def pool_finish(tasks, slots):
    """
    Tasks have durations and run in order, using at most slots at once. Return
    the finish time.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(pool_finish, [
    (([4, 2, 8, 3], 2), 10),
    (([1, 1, 1], 5), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Track each slot's free time, and scan for the earliest.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A heap of finish times.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Track each slot's free time, and scan for the earliest.

```python
def pool_finish(tasks, slots):
    free = [0] * slots
    for t in tasks:
        i = free.index(min(free))
        free[i] += t
    return max(free)
```

**Cost:** O(n·s) time, O(s) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A heap of finish times.

```python
def pool_finish(tasks, slots):
    heap = [0] * slots
    for t in tasks:
        heapq.heappush(heap, heapq.heappop(heap) + t)
    return max(heap)
```

**Cost:** O(n log s) time, O(s) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Track each slot's free time, and scan for the earliest. That costs O(n·s) time and O(s) space."

"I can do better: A heap of finish times. That's O(n log s) time and O(s) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/airflow/pool-scheduling)

<a id="q20"></a>
### Q20. A Tiny Scheduler

🔴 Stretch · *Put it all together*

Run a DAG given task results (True or False). Failed tasks block their downstream tasks ('upstream\_failed'); others run. Return state per task, sorted by name.

**Examples**

- `edges = [('a', 'b'), ('b', 'c'), ('a', 'd')], results = {'a': True, 'b': False, 'c': True, 'd': True}` → `{'a': 'success', 'b': 'failed', 'c': 'upstream_failed', 'd': 'success'}`

In [ ]:
def scheduler(edges, results):
    """
    Run a DAG given task results (True or False). Failed tasks block their
    downstream tasks ('upstream_failed'); others run. Return state per task,
    sorted by name.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(scheduler, [
    (([('a', 'b'), ('b', 'c'), ('a', 'd')], {'a': True, 'b': False, 'c': True, 'd': True}), {'a': 'success', 'b': 'failed', 'c': 'upstream_failed', 'd': 'success'}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Repeat scanning for tasks with finished upstream.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Kahn's order, tracking state.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Repeat scanning for tasks with finished upstream.

```python
def scheduler(edges, results):
    state = {}
    while len(state) < len(results):
        for t in sorted(results):
            if t in state:
                continue
            ups = [a for a, b in edges if b == t]
            if all(u in state for u in ups):
                if any(state[u] != "success" for u in ups):
                    state[t] = "upstream_failed"
                else:
                    state[t] = "success" if results[t] else "failed"
    return dict(sorted(state.items()))
```

**Cost:** O(V²) time, O(V) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Kahn's order, tracking state.

```python
def scheduler(edges, results):
    before, after = defaultdict(list), defaultdict(list)
    for a, b in edges:
        before[b].append(a)
        after[a].append(b)
    waiting = {t: len(before[t]) for t in results}
    ready = deque(t for t, n in waiting.items() if n == 0)
    state = {}
    while ready:
        t = ready.popleft()
        blocked = any(state[u] != "success" for u in before[t])
        state[t] = "upstream_failed" if blocked else ("success" if results[t] else "failed")
        for m in after[t]:
            waiting[m] -= 1
            if not waiting[m]:
                ready.append(m)
    return dict(sorted(state.items()))
```

**Cost:** O(V + E) time, O(V) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Repeat scanning for tasks with finished upstream. That costs O(V²) time and O(V) space."

"I can do better: Kahn's order, tracking state. That's O(V + E) time and O(V) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/airflow/a-tiny-scheduler)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. What is a DAG, and why must it be acyclic?</b></summary>

A graph of tasks with directed dependencies. A cycle would mean tasks waiting on each other forever, so no valid order exists.

</details>

<details>
<summary><b>2. What is the logical date in Airflow?</b></summary>

The start of the data interval a run covers, not when it runs. A daily run for 5 January actually starts when the interval ends, on 6 January.

</details>

<details>
<summary><b>3. What's catchup?</b></summary>

Airflow creates runs for all past intervals since the start date. Turn it off unless you want that.

</details>

<details>
<summary><b>4. Why not put heavy work in Airflow tasks?</b></summary>

Airflow is a coordinator. Heavy work belongs in Spark or warehouses, which scale, while workers stay light and reliable.

</details>

<details>
<summary><b>5. What should you avoid in XCom?</b></summary>

Large data, since it goes in the metadata database. Store files in object storage, and pass the path.

</details>

<details>
<summary><b>6. How do you make Airflow tasks idempotent?</b></summary>

Use the logical date for parameters, and overwrite or upsert outputs, so retries and backfills give the same result.

</details>

<details>
<summary><b>7. Scenario: a task is stuck in 'queued'. What do you check?</b></summary>

Worker availability, pool and concurrency limits, and the executor's health. Also look at the scheduler's logs.

</details>

<details>
<summary><b>8. Scenario: the DAG ran at the time you expected, but processed the wrong day. Why?</b></summary>

It probably used the current date instead of the logical date, or misunderstood that runs occur after their interval ends. Use ds or the data interval.

</details>

<details>
<summary><b>9. Scenario: after deployment, hundreds of runs started at once. What happened?</b></summary>

Catchup was on with an old start date. Pause the DAG, clear runs as needed, set catchup=False and limit max\_active\_runs.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Why you need an orchestrator
- [ ] I can explain: Dates: the idea that confuses everyone
- [ ] I can explain: Failures, retries and dependencies
- [ ] I can explain: Airflow mistakes and alternatives

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Backfill With a Concurrency Limit
- [ ] [Q17](#q17) Duration Anomaly
- [ ] [Q18](#q18) Critical Path
- [ ] [Q19](#q19) Pool Scheduling
- [ ] [Q20](#q20) A Tiny Scheduler

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.